# CV berurutan waktu (forward chaining)
**tabular › validation** · notebook `time_split_cv`

**Kapan dipakai:** Data punya dimensi waktu dan test ada di masa depan. Latih di masa lalu, uji di masa berikutnya.

**Jebakan umum:** Jangan pernah mengacak data berurutan waktu. Fitur yang tidak akan tersedia saat prediksi (data masa depan) harus dibuang.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi
Ganti bagian ini saja saat memakai data lomba.

In [ ]:
TRAIN_PATH = ROOT / "data_latihan/tabular_extra/grouped/mpg_train.csv"
TEST_PATH  = ROOT / "data_latihan/tabular_extra/grouped/mpg_test.csv"      # None kalau tidak ada file test
TARGET     = "mpg"
ID_COL     = 'id'
DROP_COLS  = ['name']
GROUP_COL  = None                      # isi bila ada grup agar tidak bocor
SCORING    = "neg_root_mean_squared_error"
SEED, N_SPLITS, VALID_SIZE = 42, 5, 0.2
MODEL_KEY  = "time_split_cv"
TASK       = "regression"
KIND       = "tree"
OUT = out_dir("tabular", "validation", MODEL_KEY)
set_seed(SEED)

## 2. Baca data

In [ ]:
df = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH) if TEST_PATH else None
print(df.shape, None if test is None else test.shape)
display(df.head(3))

## 3. Fitur & split

In [ ]:
from superdb.tabular import detect_columns, prepare_features, build_pipeline, split_train_valid, cv_score
cols = detect_columns(df, target=TARGET, id_col=ID_COL, drop_cols=DROP_COLS)
X, NUM, CAT = prepare_features(df, cols)
y = df[TARGET].to_numpy(dtype=float)
groups = df[GROUP_COL] if GROUP_COL else None
Xtr, Xva, ytr, yva = split_train_valid(X, y, TASK, VALID_SIZE, SEED, groups)
GTR = None if groups is None else groups.iloc[Xtr.index]   # grup untuk CV
print(Xtr.shape, Xva.shape)

## 4. Forward chaining
Urutkan menurut waktu (`model_year`), lalu tiap fold: latih pada masa lalu, uji pada periode setelahnya. Jangan mengacak data berurutan.

In [ ]:
from lightgbm import LGBMRegressor
from sklearn.model_selection import KFold, TimeSeriesSplit
TIME_COL = "model_year"
order = np.argsort(df[TIME_COL].to_numpy(), kind="stable")
Xo, yo, to = X.iloc[order].reset_index(drop=True), y[order], df[TIME_COL].to_numpy()[order]
pp = build_pipeline(LGBMRegressor(n_estimators=300, learning_rate=0.05, num_leaves=15, random_state=SEED, n_jobs=-1, verbose=-1), NUM, CAT, kind="tree")
rows = []
for k, (tr, va) in enumerate(TimeSeriesSplit(n_splits=4).split(Xo)):
    pp.fit(Xo.iloc[tr], yo[tr]); r = regression_metrics(yo[va], pp.predict(Xo.iloc[va]))
    rows.append({"fold": k, "latih": f"{to[tr].min()}-{to[tr].max()}", "uji": f"{to[va].min()}-{to[va].max()}", "rmse": r["rmse"]})
res_t = pd.DataFrame(rows); display(res_t.round(3))
from sklearn.model_selection import cross_val_score
rmse_rand = -cross_val_score(pp, Xo, yo, cv=KFold(5, shuffle=True, random_state=SEED), scoring="neg_root_mean_squared_error").mean()
print(f"RMSE CV berurutan waktu = {res_t.rmse.mean():.3f} | RMSE CV acak (bocor dari masa depan) = {rmse_rand:.3f}")

CV acak biasanya tampak lebih baik karena model 'mengintip' masa depan. Bila test lomba ada di periode setelah train, angka berurutan waktu-lah yang realistis.

In [ ]:
save_json({"rmse_time_cv": float(res_t.rmse.mean()), "rmse_random_cv": float(rmse_rand), "folds": res_t.to_dict("records")}, OUT / "metrics.json"); print("OK. hasil di", OUT)

## Langkah berikutnya
- Pilih skema validasi SEBELUM bereksperimen, dan pakai terus untuk semua model agar skor bisa dibandingkan.
- Jika skor CV dan skor leaderboard tidak sejalan, curigai skema validasi, bukan modelnya.